# Euro2024 Multi-Match Chain Pipeline (clean rewrite)

Этот ноутбук переписан под твой сценарий:
- сначала работаем со **стандартизированными JSON**;
- если для матча стандартизированных файлов нет — создаём и сохраняем;
- после выгрузки делаем визуальную проверку событий (как раньше);
- затем формируем цепочки, строим признаки и обучаем модели.

## Что делаем в этой ячейке

Импортируем библиотеки и готовые helper-модули `statsbomb_toolkit.sber_exports`. В этом ноутбуке стандартизацию заново не запускаем: все JSON уже лежат в `outputs/euro2024_all/processed_json`.

In [29]:
from pathlib import Path
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, confusion_matrix

from statsbomb_toolkit.sber_exports import preprocessing, viz

try:
    from catboost import CatBoostClassifier, CatBoostRanker
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    HAS_TABNET = True
except Exception:
    HAS_TABNET = False

sns.set_theme(style='whitegrid', context='talk')

## Что делаем в этой ячейке

Задаём пути и параметры эксперимента. Главный вход — готовый `index_all.csv` с путями на `events_std`, `events_std_clean`, `sb360_std`, `bad_ids`, `llm_items_jsonl` и `meta`.

In [30]:
# --- Paths ---
ROOT = Path.cwd()
INDEX_ALL = ROOT / 'outputs' / 'euro2024_all' / 'processed_json' / 'index_all.csv'
INDEX_TOP15 = ROOT / 'outputs' / 'euro2024_all' / 'processed_json' / 'index_to_speak_top15.csv'

OUT_DIR = ROOT / 'outputs' / 'euro2024_multimatch_clean'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DIR = ROOT / 'outputs' / 'euro2024_all' / 'processed_json'
if not PROCESSED_DIR.exists():
    raise FileNotFoundError(f'Не найдена папка: {PROCESSED_DIR}')

FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- Behavior ---
USE_TOP15_ONLY = False
RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20

# chain limits (короткие цепочки)
MAX_EVENTS_PER_CHAIN = 6
MAX_CHAIN_DURATION_SEC = 10.0
MAX_GAP_SEC = 2.5

# LLM export policy
MIN_SOFT_LABEL_FOR_LLM = 1  # keep soft_label_3 >= 1

## Что делаем в этой ячейке

Читаем готовый index CSV. Если `USE_TOP15_ONLY=True`, берём только TOP-15 матчей для озвучки; иначе все 51 матч.

In [31]:
# Грузим готовый индекс файлов (уже с абсолютными путями на json/jsonl)
idx_path = INDEX_TOP15 if USE_TOP15_ONLY else INDEX_ALL
if not idx_path.exists():
    raise FileNotFoundError(f'Не найден index CSV: {idx_path}')

manifest = pd.read_csv(idx_path)
manifest['match_id'] = manifest['match_id'].astype(str)
manifest = manifest.sort_values('match_id').reset_index(drop=True)

required_cols = {
    'match_id', 'to_speak', 'events_std', 'events_std_clean',
    'sb360_std', 'bad_ids', 'llm_items_jsonl', 'meta'
}
missing = sorted(list(required_cols - set(manifest.columns)))
if missing:
    raise ValueError(f'В index CSV нет обязательных колонок: {missing}')

print('index source:', idx_path)
print('matches selected:', len(manifest))
display(manifest.head(10))

index source: /Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/index_all.csv
matches selected: 51


,match_id,to_speak,events_std,events_std_clean,sb360_std,bad_ids,llm_items_jsonl,meta
0,3930158,1,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930158_events_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930158_events_std_clean.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930158_sb360_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930158_bad_ids.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930158_llm_items.jsonl,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930158_meta.json
1,3930159,0,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930159_events_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930159_events_std_clean.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930159_sb360_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930159_bad_ids.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930159_llm_items.jsonl,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930159_meta.json
2,3930160,0,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930160_events_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930160_events_std_clean.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930160_sb360_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930160_bad_ids.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930160_llm_items.jsonl,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930160_meta.json
3,3930161,0,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930161_events_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930161_events_std_clean.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930161_sb360_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930161_bad_ids.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930161_llm_items.jsonl,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930161_meta.json
4,3930162,0,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930162_events_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930162_events_std_clean.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930162_sb360_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930162_bad_ids.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930162_llm_items.jsonl,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930162_meta.json
5,3930163,0,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930163_events_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930163_events_std_clean.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930163_sb360_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930163_bad_ids.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930163_llm_items.jsonl,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930163_meta.json
6,3930164,0,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930164_events_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930164_events_std_clean.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930164_sb360_std.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_all/processed_json/3930164_bad_ids.json,/Users/angelina23/Documents/ml-ami/outputs/euro2024_

## Что делаем в этой ячейке
Определяем утилиты для загрузки/сохранения стандартизированных артефактов на матч.

In [32]:
def _load_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))


def _save_json(path: Path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')


def _as_path(x):
    return Path(str(x))

## Что делаем в этой ячейке

Загружаем готовые JSON по путям из index CSV. Никакой повторной стандартизации здесь нет: только чтение файлов и сбор `prepared_by_match`.

In [33]:
summary_rows = []
prepared_by_match = {}

# Ручные override для half-aware ref (если нужны точечно)
REFERENCE_OVERRIDE_BY_MATCH = {
    # '3930158': {1: 'Scotland', 2: 'Germany'},
}

for _, row in manifest.iterrows():
    mid = str(row['match_id'])

    p_events_std = _as_path(row['events_std'])
    p_events_std_clean = _as_path(row['events_std_clean'])
    p_sb360_std = _as_path(row['sb360_std'])
    p_bad_ids = _as_path(row['bad_ids'])
    p_meta = _as_path(row['meta'])

    needed = [p_events_std_clean, p_sb360_std, p_bad_ids, p_meta]
    missing_files = [str(x) for x in needed if not x.exists()]
    if missing_files:
        raise FileNotFoundError(f'match_id={mid}: отсутствуют файлы: {missing_files}')

    events_std_clean = _load_json(p_events_std_clean)
    events_std = _load_json(p_events_std) if p_events_std.exists() else events_std_clean
    sb360_std = _load_json(p_sb360_std)
    bad_ids = set(_load_json(p_bad_ids))
    meta = _load_json(p_meta)

    # ref_by_period: override -> meta -> fallback
    ref_by_period = REFERENCE_OVERRIDE_BY_MATCH.get(mid)
    if ref_by_period is None:
        raw_ref = meta.get('ref_by_period')
        if isinstance(raw_ref, dict):
            ref_by_period = {}
            for k, v in raw_ref.items():
                try:
                    ref_by_period[int(k)] = v
                except Exception:
                    pass

    if not isinstance(ref_by_period, dict) or 1 not in ref_by_period or 2 not in ref_by_period:
        team_names = []
        for ev in events_std:
            t = (ev.get('team') or {}).get('name')
            if t and t not in team_names:
                team_names.append(t)
            if len(team_names) >= 2:
                break
        if len(team_names) >= 2:
            ref_by_period = {1: team_names[0], 2: team_names[1]}
        elif len(team_names) == 1:
            ref_by_period = {1: team_names[0], 2: team_names[0]}
        else:
            ref_by_period = {1: 'TeamA', 2: 'TeamB'}

    teams_meta = meta.get('teams')
    if isinstance(teams_meta, list) and len(teams_meta) >= 2:
        teams_str = f"{teams_meta[0]} vs {teams_meta[1]}"
    else:
        teams_str = 'unknown vs unknown'

    prepared_by_match[mid] = {
        'events_std': events_std,
        'events_std_clean': events_std_clean,
        'sb360_std': sb360_std,
        'bad_ids': bad_ids,
        'meta': meta,
        'ref_by_period': ref_by_period,
    }

    summary_rows.append({
        'match_id': mid,
        'stage': meta.get('stage', 'EURO 2024'),
        'teams': teams_str,
        'events_n': len(events_std_clean),
        'sb360_n': len(sb360_std),
        'bad_ids_n': len(bad_ids),
        'to_speak': int(row.get('to_speak', meta.get('to_speak', 0))),
    })

std_summary = pd.DataFrame(summary_rows).sort_values('match_id').reset_index(drop=True)
std_summary.to_csv(OUT_DIR / 'loaded_bundles_index.csv', index=False)
print('loaded bundles:', len(std_summary))
print('saved index:', OUT_DIR / 'loaded_bundles_index.csv')
display(std_summary.head(12))

loaded bundles: 51
saved index: /Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/loaded_bundles_index.csv


,match_id,stage,teams,events_n,sb360_n,bad_ids_n,to_speak
0,3930158,EURO 2024,Germany vs Scotland,3373,3088,133,1
1,3930159,EURO 2024,Hungary vs Switzerland,3356,3017,163,0
2,3930160,EURO 2024,Spain vs Croatia,3796,3237,173,0
3,3930161,EURO 2024,Italy vs Albania,4066,3462,96,0
4,3930162,EURO 2024,Slovenia vs Denmark,3509,3120,196,0
5,3930163,EURO 2024,Serbia vs England,3920,3276,157,0
6,3930164,EURO 2024,Belgium vs Slovakia,3543,3153,173,0
7,3930165,EURO 2024,Austria vs France,3718,3333,231,0
8,3930166,EURO 2024,Portugal vs Czech Republic,3602,2960,211,0
9,3930167,EURO 2024,Croatia vs Albania,3721,3307,167,0


## Что делаем в этой ячейке
Делаем визуальную проверку после стандартизации: несколько эпизодов 1-го и 2-го тайма для каждого матча.

Визуализация использует тот же стиль, что раньше. `sb360` отключается только для `bad_ids`.

In [34]:
VIS_EXAMPLES_DIR = FIG_DIR / 'post_standardization_examples'
VIS_EXAMPLES_DIR.mkdir(parents=True, exist_ok=True)

TEAM_COLORS_DEFAULT = viz.DEFAULT_TEAM_COLORS.copy()
TEAM_COLORS_DEFAULT.update({
    'Georgia': '#2563eb',  # чтобы не сливалась с Spain
    'Spain': '#c62828',
})

KEY_VIS_TYPES = {
    'Pass', 'Carry', 'Shot', 'Goal Keeper', 'Interception', 'Ball Recovery',
    'Dispossessed', 'Miscontrol', 'Foul Won', 'Foul Committed', 'Offside', 'Clearance'
}

vis_rows = []
for mid, pack in prepared_by_match.items():
    events_std = pack['events_std']
    sb360_std = pack['sb360_std']
    bad_ids = pack['bad_ids']
    ref_by_period = pack['ref_by_period']

    jersey_by_id, jersey_by_name = viz.build_jersey_maps(events_std)
    df_ev = viz.make_viz_events_df(events_std, jersey_by_id=jersey_by_id, jersey_by_name=jersey_by_name)
    frames_by_id, visible_by_id = viz.make_360_indexes(sb360_std)

    for period in [1, 2]:
        sub = df_ev[df_ev['period'] == period].copy()
        if sub.empty:
            continue

        key = sub[sub['type_name'].isin(KEY_VIS_TYPES) & sub['event_x'].notna()]
        key_ids = key['id'].tolist()[:2]

        bad_sub = sub[sub['id'].isin(bad_ids) & sub['event_x'].notna()]
        bad_ids_pick = bad_sub['id'].tolist()[:1]

        for kind, ids in [('key', key_ids), ('bad', bad_ids_pick)]:
            for i, eid in enumerate(ids, start=1):
                local_frames = {} if (kind == 'bad') else frames_by_id
                local_visible = {} if (kind == 'bad') else visible_by_id

                fig, ax = viz.draw_event_keep_style_half_switch_full(
                    df_ev,
                    eid,
                    local_frames,
                    local_visible,
                    ref_by_period=ref_by_period,
                    team_colors=TEAM_COLORS_DEFAULT,
                    show_badge=True,
                    show_header=True,
                    show_ref=False,
                    show_actor_source=False,
                )

                out_png = VIS_EXAMPLES_DIR / f'{mid}_period{period}_{kind}_{i}.png'
                fig.savefig(out_png, dpi=130, bbox_inches='tight')
                plt.close(fig)

                vis_rows.append({
                    'match_id': mid,
                    'period': period,
                    'kind': kind,
                    'event_id': eid,
                    'png': str(out_png),
                })

vis_df = pd.DataFrame(vis_rows)
vis_df.to_csv(VIS_EXAMPLES_DIR / 'index.csv', index=False)
print('saved visuals:', len(vis_df), '->', VIS_EXAMPLES_DIR / 'index.csv')
display(vis_df.head(12))

saved visuals: 306 -> /Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/figures/post_standardization_examples/index.csv


,match_id,period,kind,event_id,png
0,3930158,1,key,df6f6fb7-f0b0-4561-9171-f4172f5e97e7,/Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/figures/post_standardization_examples/3930158_p...
1,3930158,1,key,65072c25-9c61-441e-a51c-52856c51ca94,/Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/figures/post_standardization_examples/3930158_p...
2,3930158,1,bad,d565ae37-8197-4510-b01f-171a04435ec1,/Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/figures/post_standardization_examples/3930158_p...
3,3930158,2,key,63034695-2cac-4d90-aac7-55b9d8b01950,/Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/figures/post_standardization_examples/3930158_p...
4,3930158,2,key,e409a9f3-ddc5-43ba-9d4e-b22ef517cdd0,/Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/figures/post_standardization_examples/3930158_p...
5,3930158,2,bad,a8311aad-ad86-4b66-b051-30739db9af8e,/Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/figures/post_standardization_examples/3930158_p...
6,3930159,1,key,ea6e1f6e-a906-43b6-a12d-bdf03af0768a,/Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/figures/post_standardization_examples/3930159_p...
7,3930159,1,key,68ca9783-809d-4d18-baec-01b0d11ed157,/Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/figures/post_standardization_examples/3930159_p...
8,3930159,1,bad,2e478a29-c69f-49be-9d5d-4893082dc841,/Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/figures/post_standardization_examples/3930159_p...
9,3930159,2,key,4e15cd49-a5a0-4f54-9292-55c670f42343,/Users/angelina23/Documents/ml-ami/outputs/euro2024_multimatch_clean/figures/post_standardization_examples/3930159_p...


## Методология: как формируем цепочки событий

На этом шаге мы не отправляем каждое событие в LLM отдельно. Сначала объединяем события в короткие футбольные эпизоды.

Идея такая:

1. **Берём уже стандартизированные события** из `outputs/euro2024_all/processed_json`. Здесь координаты уже приведены к выбранной ориентации тайма, плохие `360` уже найдены, а очищенные события лежат отдельно в `events_std_clean`.
2. **Строим двусторонний граф `related_events`**. В StatsBomb связь может идти не только `Pass -> Ball Receipt*`, но и в обратную сторону через другие события. Поэтому используем и outgoing, и incoming связи.
3. **Компонента связности = базовая related-цепочка**. Например, пас, приём, carry, pressure, block могут попасть в одну компоненту.
4. **Режем слишком длинные компоненты**. Это критично: если отдать LLM 40-80 событий, она начинает пересказывать всё подряд и терять главный смысл. Поэтому вводим ограничения:
   - `MAX_EVENTS_PER_CHAIN = 6`,
   - `MAX_CHAIN_DURATION_SEC = 10`,
   - `MAX_GAP_SEC = 2.5`.
5. **Сохраняем порядок по `index`**. Цепочка остаётся хронологической.

Почему так: `related_events` хорошо склеивает технически связанные события, но сам по себе может давать слишком длинные компоненты. Поэтому итоговая единица для модели и LLM — это **короткий эпизод**, а не весь related-компонент.


In [35]:
def build_related_graph(events_std):
    events_by_id = {e.get('id'): e for e in events_std if e.get('id')}
    outgoing, incoming = {}, {}

    for e in events_std:
        eid = e.get('id')
        if not eid:
            continue
        rel = e.get('related_events') or []
        outgoing.setdefault(eid, set())
        for rid in rel:
            if rid:
                outgoing[eid].add(rid)
                incoming.setdefault(rid, set()).add(eid)

    neighbors = {eid: set() for eid in events_by_id}
    for eid in events_by_id:
        neighbors[eid] |= outgoing.get(eid, set())
        neighbors[eid] |= incoming.get(eid, set())
    return events_by_id, neighbors


def build_compact_chains(events_std, max_events=6, max_duration_sec=10.0, max_gap_sec=2.5):
    events_by_id, neighbors = build_related_graph(events_std)
    all_ids = set(events_by_id)

    def expand(root_id, used):
        group = set()
        stack = [root_id]
        while stack:
            cur = stack.pop()
            if not cur or cur in group:
                continue
            if cur in used and cur != root_id:
                continue
            group.add(cur)
            for nb in neighbors.get(cur, set()):
                if nb in group or nb in used:
                    continue
                if nb in all_ids:
                    stack.append(nb)
        return group

    ev_sorted = sorted(events_std, key=lambda e: (e.get('index', 10**9), e.get('id') or ''))
    used = set()
    chains = []

    for e in ev_sorted:
        eid = e.get('id')
        if not eid or eid in used:
            continue

        group = expand(eid, used)
        chain = sorted(group, key=lambda x: events_by_id.get(x, {}).get('index', 10**9))

        # ограничение длины/времени/гепа (режем длинные цепочки)
        chunk = []
        t0 = None
        prev_t = None

        def _sec(ts):
            if not isinstance(ts, str):
                return None
            import re
            m = re.match(r'^(\d+):(\d+):(\d+)(?:\.(\d+))?$', ts)
            if not m:
                return None
            hh, mm, ss, ms = m.groups()
            base = int(hh)*3600 + int(mm)*60 + int(ss)
            frac = float(f"0.{ms}") if ms else 0.0
            return base + frac

        for cid in chain:
            ev = events_by_id[cid]
            ts = ev.get('timestamp')
            t = _sec(ts)

            if not chunk:
                chunk = [cid]
                t0 = t
                prev_t = t
                continue

            need_cut = False
            if len(chunk) >= max_events:
                need_cut = True
            if t is not None and t0 is not None and (t - t0) > max_duration_sec:
                need_cut = True
            if t is not None and prev_t is not None and (t - prev_t) > max_gap_sec:
                need_cut = True

            if need_cut:
                chains.append(chunk)
                used.update(chunk)
                chunk = [cid]
                t0 = t
                prev_t = t
            else:
                chunk.append(cid)
                prev_t = t

        if chunk:
            chains.append(chunk)
            used.update(chunk)

    return chains, events_by_id

## Методология: где появляются зоны, движение и LLM-payload

В этой ячейке мы превращаем короткие цепочки в `payload` для LLM и обучения.

Важно: зоны не считаются вручную в этой ячейке. Они считаются внутри функции:

`statsbomb_toolkit.sber_exports.preprocessing.build_chain_payload_v4`

Эта функция добавляет к каждому событию блок `derived`:

- `derived.orientation` — какая команда где защищается и куда атакует в текущем тайме;
- `derived.zones` — абсолютные и командно-относительные зоны поля;
- `derived.movement` — движение вперёд/назад/поперёк относительно команды с мячом;
- `derived.episode_signals` — сигналы для отбора: длинный пас, вход в чужую половину, вход в штрафную, смена фланга, стиль паса.

То есть для CatBoost эти поля становятся табличными признаками, а для LLM — объяснимым футбольным контекстом. LLM не должна сама вычислять геометрию по координатам: она получает уже готовые признаки.

Отдельно: если событие попало в `bad_ids`, то `sb360_json` в payload не передаётся. Это защищает LLM от неверного freeze-frame.


In [36]:
chains_all = []
payloads_all = []

for mid, pack in prepared_by_match.items():
    chains_mid, events_by_id = build_compact_chains(
        pack['events_std'],
        max_events=MAX_EVENTS_PER_CHAIN,
        max_duration_sec=MAX_CHAIN_DURATION_SEC,
        max_gap_sec=MAX_GAP_SEC,
    )

    events_clean_by_id = {e.get('id'): ce for e, ce in zip(pack['events_std'], pack['events_std_clean']) if e.get('id')}
    sb360_by_id = {(s.get('event_uuid') or s.get('id')): s for s in pack['sb360_std'] if (s.get('event_uuid') or s.get('id'))}

    for ch in chains_mid:
        payload = preprocessing.build_chain_payload_v4(
            ch,
            events_by_id=events_by_id,
            events_clean_by_id=events_clean_by_id,
            sb360_by_id=sb360_by_id,
            bad_ids=set(pack['bad_ids']),
            ref_by_period=pack['ref_by_period'],
        )
        payload['match_id'] = mid
        payload['stage'] = pack['meta'].get('stage') or 'EURO 2024'
        payload['to_speak'] = int(pack['meta'].get('to_speak', 0))
        payloads_all.append(payload)
        chains_all.append(ch)

print('chains total:', len(chains_all))
print('payloads total:', len(payloads_all))

chains total: 49300
payloads total: 49300


## Проверка: примеры получившихся цепочек

После формирования `payloads_all` полезно руками посмотреть несколько цепочек.

Эта ячейка показывает:

- короткую сводку по цепочке: время, тип события, команда, игрок, зоны, движение, наличие 360;
- полный JSON payload, который потом можно отправить в LLM;
- отдельно можно выбрать короткие, средние или длинные цепочки.

Так проще проверить, что `Pass + Ball Receipt* + Carry` действительно склеиваются, а слишком длинные эпизоды режутся.


In [37]:
# --- Chain examples viewer ---
# Запусти эту ячейку после формирования payloads_all.

import json


def _safe_name(obj):
    if isinstance(obj, dict):
        return obj.get('name')
    return obj


def summarize_payload_chain(payload):
    rows = []
    for pos, item in enumerate(payload.get('events', []), start=1):
        ej = item.get('event_json') or {}
        d = item.get('derived') or {}
        zones = d.get('zones') or {}
        mov = d.get('movement') or {}
        sig = d.get('episode_signals') or {}
        rows.append({
            'pos': pos,
            'event_id': ej.get('id'),
            'index': ej.get('index'),
            'period': ej.get('period'),
            'timestamp': ej.get('timestamp'),
            'type': _safe_name(ej.get('type')),
            'team': _safe_name(ej.get('team')),
            'player': _safe_name(ej.get('player')),
            'recipient': sig.get('pass_recipient'),
            'start_rel': zones.get('start_rel'),
            'end_rel': zones.get('end_rel'),
            'start_abs': zones.get('start_abs'),
            'end_abs': zones.get('end_abs'),
            'movement': mov.get('label'),
            'compass': mov.get('compass'),
            'forward_delta': mov.get('forward_delta'),
            'is_long_pass': sig.get('is_long_pass'),
            'entered_box': sig.get('entered_opponent_box'),
            'switched_flank': sig.get('switched_flank'),
            'has_360': item.get('sb360_json') is not None,
        })
    return pd.DataFrame(rows)


def show_chain_examples(payloads, k=5, mode='mixed', full_json=False):
    # mode: short / medium / long / important / mixed
    if not payloads:
        print('payloads пустой')
        return

    enriched = []
    for i, p in enumerate(payloads, start=1):
        n = len(p.get('events', []))
        score = (p.get('chain_features') or {}).get('importance_score_rule', 0)
        enriched.append((i, n, score, p))

    if mode == 'short':
        chosen = [x for x in enriched if x[1] <= 2][:k]
    elif mode == 'medium':
        chosen = [x for x in enriched if 3 <= x[1] <= 6][:k]
    elif mode == 'long':
        chosen = sorted(enriched, key=lambda x: x[1], reverse=True)[:k]
    elif mode == 'important':
        chosen = sorted(enriched, key=lambda x: x[2], reverse=True)[:k]
    else:
        short = [x for x in enriched if x[1] <= 2][:max(1, k//3)]
        medium = [x for x in enriched if 3 <= x[1] <= 6][:max(1, k//3)]
        long = sorted(enriched, key=lambda x: x[1], reverse=True)[:max(1, k - len(short) - len(medium))]
        chosen = short + medium + long

    for chain_idx, n, score, p in chosen:
        print()
        print('=' * 100)
        print(f"CHAIN #{chain_idx} | match_id={p.get('match_id')} | events={n} | importance_score_rule={score}")
        print('chain_event_ids:', p.get('chain_event_ids'))
        print('chain_features:', json.dumps(p.get('chain_features') or {}, ensure_ascii=False))
        display(summarize_payload_chain(p))
        if full_json:
            print(json.dumps(p, ensure_ascii=False, indent=2))


# Примеры для просмотра.
show_chain_examples(payloads_all, k=6, mode='mixed', full_json=False)

# Если нужен полный JSON конкретной цепочки:
# chain_no = 1
# print(json.dumps(payloads_all[chain_no - 1], ensure_ascii=False, indent=2))

# Другие режимы:
# show_chain_examples(payloads_all, k=5, mode='short')
# show_chain_examples(payloads_all, k=5, mode='medium')
# show_chain_examples(payloads_all, k=5, mode='long')
# show_chain_examples(payloads_all, k=5, mode='important')



CHAIN #1 | match_id=3930158 | events=1 | importance_score_rule=0
chain_event_ids: ['f2207d37-beed-4073-84a5-f9d696e2afdd']
chain_features: {"events_n": 1, "high_events_n": 0, "set_play_events_n": 0, "low_events_n": 0, "entered_opponent_box_n": 0, "importance_score_rule": 0}


,pos,event_id,index,period,timestamp,type,team,player,recipient,start_rel,end_rel,start_abs,end_abs,movement,compass,forward_delta,is_long_pass,entered_box,switched_flank,has_360
0,1,f2207d37-beed-4073-84a5-f9d696e2afdd,1,1,00:00:00.000,Starting XI,Germany,None,None,None,None,None,None,None,None,None,None,None,None,False



CHAIN #2 | match_id=3930158 | events=1 | importance_score_rule=0
chain_event_ids: ['43e494fe-4871-4b2a-a509-a325c17a5e3f']
chain_features: {"events_n": 1, "high_events_n": 0, "set_play_events_n": 0, "low_events_n": 0, "entered_opponent_box_n": 0, "importance_score_rule": 0}


,pos,event_id,index,period,timestamp,type,team,player,recipient,start_rel,end_rel,start_abs,end_abs,movement,compass,forward_delta,is_long_pass,entered_box,switched_flank,has_360
0,1,43e494fe-4871-4b2a-a509-a325c17a5e3f,2,1,00:00:00.000,Starting XI,Scotland,None,None,None,None,None,None,None,None,None,None,None,None,False



CHAIN #4 | match_id=3930158 | events=4 | importance_score_rule=3
chain_event_ids: ['df6f6fb7-f0b0-4561-9171-f4172f5e97e7', 'f9195e75-1617-478c-bf9d-66f17151724a', '65072c25-9c61-441e-a51c-52856c51ca94', '01be895c-5835-4a8d-b99f-2a642f5d26dc']
chain_features: {"events_n": 4, "high_events_n": 0, "set_play_events_n": 4, "low_events_n": 1, "entered_opponent_box_n": 0, "importance_score_rule": 3}


...

,pos,event_id,index,period,timestamp,type,team,player,recipient,start_rel,end_rel,start_abs,end_abs,movement,compass,forward_delta,is_long_pass,entered_box,switched_flank,has_360
0,1,d565ae37-8197-4510-b01f-171a04435ec1,10,1,00:00:07.397,Duel,Scotland,Hendry,None,own_half,NaN,pre_box_left,NaN,unknown,unknown,NaN,0,0,1,False
1,2,859a97e2-e05a-42c3-a020-32a8d7848300,11,1,00:00:07.397,Pass,Germany,Havertz,None,opponent_half,opponent_half,pre_box_left,pre_box_left,short_or_static,short_or_static,-2.1,0,0,0,True
2,3,1b105dba-9074-4a8d-b6cd-ab686d59d32e,12,1,00:00:07.871,Ball Recovery,Scotland,McGregor,None,own_half,NaN,pre_box_left,NaN,unknown,unknown,NaN,0,0,1,True



CHAIN #7 | match_id=3930158 | events=6 | importance_score_rule=8
chain_event_ids: ['ed432afe-033b-418b-b366-db95e3b993f0', '44ed9025-a8bd-4455-8a0d-404ddd5e926f', '8226f5a6-a6ef-49cf-b3a7-24146fe8a028', 'ce958de1-f077-4032-9604-3ed4b57cea3b', '4484cb90-4998-48cd-a0be-472c4be2a2a2', '7040ea80-1599-41ef-9593-0aa783085750']
chain_features: {"events_n": 6, "high_events_n": 1, "set_play_events_n": 6, "low_events_n": 2, "entered_opponent_box_n": 0, "importance_score_rule": 8}


,pos,event_id,index,period,timestamp,type,team,player,recipient,start_rel,end_rel,start_abs,end_abs,movement,compass,forward_delta,is_long_pass,entered_box,switched_flank,has_360
0,1,ed432afe-033b-418b-b366-db95e3b993f0,13,1,00:00:08.713,Ball Recovery,Scotland,Hendry,None,own_half,NaN,left_box,NaN,unknown,unknown,NaN,0,0,1,True
1,2,44ed9025-a8bd-4455-8a0d-404ddd5e926f,14,1,00:00:08.713,Carry,Scotland,Hendry,None,own_half,own_half,left_box,pre_box_left,lateral,left,1.4,0,0,0,True
2,3,8226f5a6-a6ef-49cf-b3a7-24146fe8a028,15,1,00:00:08.998,Pressure,Germany,Wirtz,None,opponent_half,NaN,left_box,NaN,unknown,unknown,NaN,0,0,1,False
3,4,ce958de1-f077-4032-9604-3ed4b57cea3b,16,1,00:00:09.173,Pressure,Germany,Musiala,None,opponent_half,NaN,pre_box_left,NaN,unknown,unknown,NaN,0,0,1,True
4,5,4484cb90-4998-48cd-a0be-472c4be2a2a2,17,1,00:00:09.544,Pass,Scotland,Hendry,None,own_half,own_half,pre_box_left,pre_box_left,short_or_static,short_or_static,1.8,0,0,0,True
5,6,7040ea80-1599-41ef-9593-0aa783085750,18,1,00:00:09.564,Block,Germany,Musiala,None,opponent_half,NaN,pre_box_left,NaN,unknown,unknown,NaN,0,0,1,True



CHAIN #10 | match_id=3930158 | events=6 | importance_score_rule=8
chain_event_ids: ['4ea84592-655a-4d31-bc76-d9938673b263', '593c9000-55dd-43c5-9494-37ade9926d48', 'b7b45596-cde9-4d8c-b794-c3e73ba4f3fd', '19a8e06c-2fb4-4128-be6f-191fd2a489dc', '42c3cd06-3162-4762-92f2-0a48b7c9eef5', 'ade698fa-53cd-4ff7-9332-b6eb4ed47ce5']
chain_features: {"events_n": 6, "high_events_n": 1, "set_play_events_n": 6, "low_events_n": 1, "entered_opponent_box_n": 0, "importance_score_rule": 8}


,pos,event_id,index,period,timestamp,type,team,player,recipient,start_rel,end_rel,start_abs,end_abs,movement,compass,forward_delta,is_long_pass,entered_box,switched_flank,has_360
0,1,4ea84592-655a-4d31-bc76-d9938673b263,21,1,00:00:13.928,Ball Recovery,Germany,Tah,NaN,own_half,NaN,center_circle_zone,NaN,unknown,unknown,NaN,0,0,1,True
1,2,593c9000-55dd-43c5-9494-37ade9926d48,22,1,00:00:13.928,Carry,Germany,Tah,NaN,own_half,own_half,center_circle_zone,center_circle_zone,lateral,left,0.7,0,0,0,True
2,3,b7b45596-cde9-4d8c-b794-c3e73ba4f3fd,23,1,00:00:15.597,Pass,Germany,Tah,Andrich,own_half,opponent_half,center_circle_zone,center_circle_zone,forward,forward,5.1,0,0,0,True
3,4,19a8e06c-2fb4-4128-be6f-191fd2a489dc,24,1,00:00:16.068,Ball Receipt*,Germany,Andrich,NaN,opponent_half,NaN,center_circle_zone,NaN,unknown,unknown,NaN,0,0,1,True
4,5,42c3cd06-3162-4762-92f2-0a48b7c9eef5,25,1,00:00:16.068,Carry,Germany,Andrich,NaN,opponent_half,own_half,center_circle_zone,center_circle_zone,backward,backward_right,-6.5,0,0,0,True
5,6,ade698fa-53cd-4ff7-9332-b6eb4ed47ce5,26,1,00:00:17.112,Pass,Germany,Andrich,Rüdiger,own_half,own_half,center_circle_zone,right_half:center_lane,lateral,right,1.3,0,0,0,True


## Проверка: какие зоны и derived-поля получает LLM

Эта ячейка не создаёт новые признаки, а помогает глазами проверить, что уже лежит в payload.

Что важно смотреть:

- `start_abs/end_abs` — абсолютная геометрия поля;
- `start_rel/end_rel` — своя/чужая зона относительно команды с мячом;
- `movement.label` и `forward_delta` — движение вперёд/назад относительно атаки команды;
- `episode_signals` — признаки, которые потом используются и для LLM, и для CatBoost.

Если здесь видно, что `start_rel/end_rel` или `forward_delta` выглядят странно, значит надо возвращаться к `ref_by_period` конкретного матча.


In [38]:
ZONE_REFERENCE_DF = pd.DataFrame([
    {'field': 'derived.orientation.own_goal_x', 'meaning': 'x-координата своих ворот команды в текущем тайме'},
    {'field': 'derived.orientation.opp_goal_x', 'meaning': 'x-координата чужих ворот команды в текущем тайме'},
    {'field': 'derived.orientation.attack_sign', 'meaning': '+1 если команда атакует вправо, -1 если атакует влево'},
    {'field': 'derived.zones.start_abs/end_abs', 'meaning': 'абсолютная зона на фиксированной карте поля'},
    {'field': 'derived.zones.start_rel/end_rel', 'meaning': 'зона относительно команды: своя/чужая половина, штрафная и т.д.'},
    {'field': 'derived.zones.start_lane/end_lane', 'meaning': 'верхний фланг / центр / нижний фланг'},
    {'field': 'derived.zones.zone_transition', 'meaning': 'переход между относительными зонами'},
    {'field': 'derived.movement.forward_delta', 'meaning': '>0 движение к чужим воротам, <0 к своим воротам'},
    {'field': 'derived.movement.label', 'meaning': 'forward/backward/lateral/short_or_static'},
    {'field': 'derived.movement.compass', 'meaning': 'направление: forward_left, backward_right и т.д.'},
    {'field': 'derived.episode_signals.is_long_pass', 'meaning': 'длинная передача/заброс'},
    {'field': 'derived.episode_signals.entered_opponent_half', 'meaning': 'вход на чужую половину'},
    {'field': 'derived.episode_signals.entered_opponent_box', 'meaning': 'вход в чужую штрафную'},
    {'field': 'derived.episode_signals.switched_flank', 'meaning': 'смена фланга/коридора'},
])
display(ZONE_REFERENCE_DF)

# Пример derived-полей из первого непустого payload.
preview_rows = []
for p in payloads_all[:200]:
    for item in p.get('events', []):
        ej = item.get('event_json') or {}
        d = item.get('derived') or {}
        if d:
            preview_rows.append({
                'match_id': p.get('match_id'),
                'chain_event_ids': p.get('chain_event_ids'),
                'timestamp': ej.get('timestamp'),
                'type': (ej.get('type') or {}).get('name'),
                'team': (ej.get('team') or {}).get('name'),
                'player': (ej.get('player') or {}).get('name'),
                'orientation': d.get('orientation'),
                'zones': d.get('zones'),
                'movement': d.get('movement'),
                'episode_signals': d.get('episode_signals'),
                'has_360': item.get('sb360_json') is not None,
            })
        if len(preview_rows) >= 10:
            break
    if len(preview_rows) >= 10:
        break

pd.set_option('display.max_colwidth', 120)
display(pd.DataFrame(preview_rows))


,field,meaning
0,derived.orientation.own_goal_x,x-координата своих ворот команды в текущем тайме
1,derived.orientation.opp_goal_x,x-координата чужих ворот команды в текущем тайме
2,derived.orientation.attack_sign,"+1 если команда атакует вправо, -1 если атакует влево"
3,derived.zones.start_abs/end_abs,абсолютная зона на фиксированной карте поля
4,derived.zones.start_rel/end_rel,"зона относительно команды: своя/чужая половина, штрафная и т.д."
5,derived.zones.start_lane/end_lane,верхний фланг / центр / нижний фланг
6,derived.zones.zone_transition,переход между относительными зонами
7,derived.movement.forward_delta,">0 движение к чужим воротам, <0 к своим воротам"
8,derived.movement.label,forward/backward/lateral/short_or_static
9,derived.movement.compass,"направление: forward_left, backward_right и т.д."


,match_id,chain_event_ids,timestamp,type,team,player,orientation,zones,movement,episode_signals,has_360
0,3930158,[f2207d37-beed-4073-84a5-f9d696e2afdd],00:00:00.000,Starting XI,Germany,NaN,None,None,None,None,False
1,3930158,[43e494fe-4871-4b2a-a509-a325c17a5e3f],00:00:00.000,Starting XI,Scotland,NaN,None,None,None,None,False
2,3930158,"[d9de8fe7-3122-4067-a1c2-a742c7edb2b8, 455f819f-3926-4ed4-a0e9-4e9e4d0bf95a]",00:00:00.000,Half Start,Germany,NaN,None,None,None,None,False
3,3930158,"[d9de8fe7-3122-4067-a1c2-a742c7edb2b8, 455f819f-3926-4ed4-a0e9-4e9e4d0bf95a]",00:00:00.000,Half Start,Scotland,NaN,None,None,None,None,False
4,3930158,"[df6f6fb7-f0b0-4561-9171-f4172f5e97e7, f9195e75-1617-478c-bf9d-66f17151724a, 65072c25-9c61-441e-a51c-52856c51ca94, 0...",00:00:01.191,Pass,Germany,Havertz,"{'period': 1, 'team': 'Germany', 'own_goal_x': 120.0, 'opp_goal_x': 0.0, 'attack_sign': -1}","{'start_abs': 'center_circle_zone', 'end_abs': 'right_half:center_lane', 'start_rel': 'center_line', 'end_rel': 'own...","{'dx': 14.299999999999997, 'dy': 11.899999999999999, 'forward_delta': -14.299999999999997, 'lateral_delta': 11.89999...","{'is_long_pass': 0, 'pass_length': 18.603764, 'pass_style_ru': 'низом', 'pass_height_name': 'Ground Pass', 'pass_rec...",True
5,3930158,"[df6f6fb7-f0b0-4561-9171-f4172f5e97e7, f9195e75-1617-478c-bf9d-66f17151724a, 65072c25-9c61-441e-a51c-52856c51ca94, 0...",00:00:02.426,Ball Receipt*,Germany,Mittelstädt,"{'period': 1, 'team': 'Germany', 'own_goal_x': 120.0, 'opp_goal_x': 0.0, 'attack_sign': -1}","{'start_abs': 'right_half:center_lane', 'end_abs': None, 'start_rel': 'own_half', 'end_rel': None, 'start_lane': 'ce...","{'dx': None, 'dy': None, 'forward_delta': None, 'lateral_delta': None, 'label': 'unknown', 'compass': 'unknown'}","{'is_long_pass': 0, 'pass_length': None, 'pass_style_ru': None, 'pass_height_name': None, 'pass_recipient': None, 'p...",True
6,3930158,"[df6f6fb7-f0b0-4561-9171-f4172f5e97e7, f9195e75-1617-478c-bf9d-66f17151724a, 65072c25-9c61-441e-a51c-52856c51ca94, 0...",00:00:02.426,Carry,Germany,Mittelstädt,"{'period': 1, 'team': 'Germany', 'own_goal_x': 120.0, 'opp_goal_x': 0.0, 'attack_sign': -1}","{'start_abs': 'right_half:center_lane', 'end_abs': 'right_half:center_lane', 'start_rel': 'own_half', 'end_rel': 'ow...","{'dx': -3.5999999999999943, 'dy': 10.0, 'forward_delta': 3.5999999999999943, 'lateral_delta': 10.0, 'label': 'forwar...","{'is_long_pass': 0, 'pass_length': None, 'pass_style_ru': None, 'pass_height_name': None, 'pass_recipient': None, 'p...",True
7,3930158,"[df6f6fb7-f0b0-4561-9171-f4172f5e97e7, f9195e75-1617-478c-bf9d-66f17151724a, 65072c25-9c61-441e-a51c-52856c51ca94, 0...",00:00:04.165,Pass,Germany,Mittelstädt,"{'period': 1, 'team': 'Germany', 'own_goal_x': 120.0, 'opp_goal_x': 0.0, 'attack_sign': -1}","{'start_abs': 'right_half:center_lane', 'end_abs': 'pre_box_left', 'start_rel': 'own_half', 'end_rel': 'opponent_hal...","{'dx': -50.60000000000001, 'dy': -26.199999999999996, 'forward_delta': 50.60000000000001, 'lateral_delta': -26.19999...","{'is_long_pass': 1, 'pass_length': 56.980698, 'pass_style_ru': 'заброс', 'pass_height_name': 'High Pass', 'pass_reci...",True
8,3930158,[9aa6c033-3015-4e13-9566-bd1913f754d3],00:00:07.397,Ball Receipt*,Germany,Havertz,"{'period': 1, 'team': 'Germany', 'own_goal_x': 120.0, 'opp_goal_x': 0.0, 'attack_sign': -1}","{'start_abs': 'pre_box_left', 'end_abs': None, 'start_rel': 'opponent_half', 'end_rel': None, 'start_lane': 'center_...","{'dx': None, 'dy': None, 'forward_delta': None, 'lateral_delta': None, 'label': 'unknown', 'compass': 'unknown'}","{'is_long_pass': 0, 'pass_length': None, 'pass_style_ru': None, 'pass_height_name': None, 'pass_recipient': None, 'p...",True
9,3930158,"[d565ae37-8197-4510-b01f-171a04435ec1, 859a97e2-e05a-42c3-a020-32a8d7848300, 1b105dba-9074-4a8d-b6cd-ab686d59d32e]",00:00:07.397,Duel,Scotland,Hendry,"{'period': 1, 'team': 'Scotland', 'own_goal_x': 0.0, 'opp_goal_x': 120.0, 'attack_sign': 1}","{'start_abs': 'pre_box_

## Методология: признаки цепочек и событий

Здесь мы переводим payload в табличные признаки для обучения.

Есть два уровня признаков.

**1. Event-level признаки** — признаки каждого события внутри цепочки:

- тип события (`Pass`, `Carry`, `Shot`, `Pressure`, ...);
- класс события: удар, переход владения, потеря, build-up, фол, low-signal и т.д.;
- зоны старта и конца (`start_abs`, `end_abs`, `start_rel`, `end_rel`);
- направление движения (`forward`, `backward`, `lateral`, `short_or_static`);
- параметры паса: длина, высота, направление, получатель, плохой исход;
- наличие/отсутствие `sb360_json` после фильтрации bad ids.

**2. Chain-level признаки** — агрегаты по всей цепочке:

- сколько событий в цепочке;
- длительность;
- сколько ударов/фолов/потерь/перехватов/стандартов;
- были ли длинные передачи;
- был ли вход в чужую половину или штрафную;
- был ли переход между третями поля;
- был ли перевод на другой фланг;
- средний и максимальный прогресс к чужим воротам;
- сколько low-signal событий (`Ball Receipt*`, `Pressure`).

Зачем столько признаков CatBoost: модель должна видеть не просто “тип первого события”, а структуру эпизода. Например, два `Pass` могут быть совершенно разными: короткий пас назад в центре и длинный заброс в штрафную. Для LLM это тоже важно, но LLM получает не таблицу, а payload с теми же derived-полями.


In [25]:
EVENT_CLASS_MAP = {
    'Pass': 'build_up', 'Carry': 'build_up', 'Dribble': 'build_up', 'Ball Receipt*': 'low_signal',
    'Pressure': 'low_signal', 'Shield': 'low_signal',
    'Interception': 'transition', 'Ball Recovery': 'transition', 'Dispossessed': 'turnover',
    'Miscontrol': 'turnover', 'Dribbled Past': 'turnover', '50/50': 'duel', 'Duel': 'duel',
    'Shot': 'shot', 'Own Goal For': 'shot_outcome', 'Own Goal Against': 'shot_outcome',
    'Goal Keeper': 'shot_outcome', 'Block': 'defensive_action', 'Clearance': 'defensive_action', 'Error': 'defensive_action',
    'Foul Committed': 'foul', 'Foul Won': 'foul', 'Offside': 'set_piece_or_ref',
    'Bad Behaviour': 'discipline', 'Referee Ball Drop': 'set_piece_or_ref', 'Referee Ball-Drop': 'set_piece_or_ref',
    'Starting XI': 'meta', 'Half Start': 'meta', 'Half End': 'meta', 'Tactical Shift': 'meta',
    'Substitution': 'meta', 'Player On': 'meta', 'Player Off': 'meta', 'Injury Stoppage': 'meta',
}

SPEC_EVENT_TYPES_33 = {
    '50/50', 'Bad Behaviour', 'Ball Receipt*', 'Ball Recovery', 'Block', 'Carry',
    'Clearance', 'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Error',
    'Foul Committed', 'Foul Won', 'Goal Keeper', 'Half End', 'Half Start',
    'Injury Stoppage', 'Interception', 'Miscontrol', 'Offside', 'Own Goal Against',
    'Own Goal For', 'Pass', 'Player Off', 'Player On', 'Pressure', 'Referee Ball Drop',
    'Shield', 'Shot', 'Starting XI', 'Substitution', 'Tactical Shift'
}

MIN_FORWARD_DELTA = 4.0


def _to_seconds(ts):
    if not isinstance(ts, str):
        return None
    import re
    m = re.match(r'^(\d+):(\d+):(\d+)(?:\.(\d+))?$', ts)
    if not m:
        return None
    hh, mm, ss, ms = m.groups()
    base = int(hh) * 3600 + int(mm) * 60 + int(ss)
    frac = float(f"0.{ms}") if ms else 0.0
    return base + frac


def event_class(t):
    return EVENT_CLASS_MAP.get(t, 'other')


def _third_from_abs(zone_abs):
    if not isinstance(zone_abs, str):
        return None
    parts = zone_abs.split(':')
    if len(parts) >= 3:
        return parts[1]
    return None


def chain_features_from_payload(payload, chain_id):
    events = payload.get('events', [])
    if not events:
        return None

    types = [((e.get('event_json') or {}).get('type') or {}).get('name') for e in events]
    classes = [event_class(t) for t in types]
    cc = Counter(classes)

    long_pass_n = entered_opp_half_n = entered_opp_box_n = 0
    left_own_box_n = low_signal_n = high_event_n = set_play_n = 0
    switched_flank_n = bad_pass_outcome_n = 0
    backward_moves_n = lateral_moves_n = progressive_carry_n = forward_moves_n = 0
    corner_zone_end_n = throwin_end_n = pre_box_end_n = 0
    third_transition_n = ended_final_third_n = center_circle_touch_n = 0

    fwd_vals, lat_vals, gaps = [], [], []
    prev_t = None

    for it in events:
        ej = it.get('event_json') or {}
        dr = it.get('derived') or {}
        sem = dr.get('event_semantics') or {}
        z = dr.get('zones') or {}
        mv = dr.get('movement') or {}
        sg = dr.get('episode_signals') or {}

        et = ((ej.get('type') or {}).get('name'))
        pp = ((ej.get('play_pattern') or {}).get('name') or '')

        if sem.get('is_low_signal', 0) == 1:
            low_signal_n += 1
        if et in {
            'Shot', 'Interception', 'Ball Recovery', 'Dispossessed', 'Miscontrol',
            'Offside', 'Foul Won', 'Foul Committed', 'Goal Keeper', 'Own Goal For',
            'Own Goal Against', 'Error', 'Block', 'Clearance'
        }:
            high_event_n += 1
        if any(k in pp.lower() for k in ['throw in','free kick','corner','goal kick','kick off']):
            set_play_n += 1

        if sg.get('is_long_pass', 0) == 1:
            long_pass_n += 1
        if sg.get('entered_opponent_half', 0) == 1:
            entered_opp_half_n += 1
        if sg.get('entered_opponent_box', 0) == 1:
            entered_opp_box_n += 1
        own_goal_x = (dr.get('orientation') or {}).get('own_goal_x')
        start_abs = z.get('start_abs')
        end_abs = z.get('end_abs')
        start_team = preprocessing.zone_label_team(start_abs, own_goal_x) if (start_abs and own_goal_x is not None) else None
        end_team = preprocessing.zone_label_team(end_abs, own_goal_x) if (end_abs and own_goal_x is not None) else None
        if start_team == 'own_box' and end_team not in {None, 'own_box'}:
            left_own_box_n += 1
        if sg.get('switched_flank', 0) == 1:
            switched_flank_n += 1

        p_out = ((ej.get('pass') or {}).get('outcome') or {}).get('name')
        if p_out in {'Incomplete','Out','Pass Offside','Unknown'}:
            bad_pass_outcome_n += 1

        fd = mv.get('forward_delta')
        ld = mv.get('lateral_delta')
        if fd is not None:
            fwd_vals.append(float(fd))
            if float(fd) > MIN_FORWARD_DELTA:
                forward_moves_n += 1
            elif float(fd) < -2:
                backward_moves_n += 1
        if ld is not None:
            lat_vals.append(abs(float(ld)))
            if abs(float(ld)) > 6 and (fd is None or abs(float(fd)) <= abs(float(ld))):
                lateral_moves_n += 1

        if et == 'Carry' and fd is not None and float(fd) > (MIN_FORWARD_DELTA / 2):
            progressive_carry_n += 1

        end_abs = z.get('end_abs')
        if end_abs == 'corner_zone':
            corner_zone_end_n += 1
        if end_abs in {'throw_in_top','throw_in_bottom'}:
            throwin_end_n += 1
        if end_abs in {'pre_box_left','pre_box_right','pre_left_box','pre_right_box'}:
            pre_box_end_n += 1

        st3 = _third_from_abs(start_abs)
        et3 = _third_from_abs(end_abs)
        if st3 and et3 and st3 != et3:
            third_transition_n += 1
        if et3 in {'left_third', 'right_third'} and own_goal_x is not None:
            opp_goal_x = (dr.get('orientation') or {}).get('opp_goal_x')
            if opp_goal_x == 120.0 and et3 == 'right_third':
                ended_final_third_n += 1
            if opp_goal_x == 0.0 and et3 == 'left_third':
                ended_final_third_n += 1
        if end_abs == 'center_circle_zone':
            center_circle_touch_n += 1

        ts = ej.get('timestamp')
        cur_t = _to_seconds(ts)
        if prev_t is not None and cur_t is not None:
            gaps.append(max(0.0, cur_t - prev_t))
        prev_t = cur_t if cur_t is not None else prev_t

    t_start = ((events[0].get('event_json') or {}).get('timestamp'))
    t_end = ((events[-1].get('event_json') or {}).get('timestamp'))
    s0, s1 = _to_seconds(t_start), _to_seconds(t_end)
    dur = (s1 - s0) if (s0 is not None and s1 is not None) else 0.0

    row = {
        'chain_id': chain_id,
        'match_id': payload.get('match_id'),
        'stage': payload.get('stage'),
        'to_speak': int(payload.get('to_speak', 0)),
        'events_n': len(events),
        'duration_sec': float(dur),
        'high_event_n': high_event_n,
        'set_play_n': set_play_n,
        'long_pass_n': long_pass_n,
        'entered_opp_half_n': entered_opp_half_n,
        'entered_opp_box_n': entered_opp_box_n,
        'left_own_box_n': left_own_box_n,
        'forward_moves_n': forward_moves_n,
        'backward_moves_n': backward_moves_n,
        'lateral_moves_n': lateral_moves_n,
        'progressive_carry_n': progressive_carry_n,
        'switched_flank_n': switched_flank_n,
        'bad_pass_outcome_n': bad_pass_outcome_n,
        'corner_zone_end_n': corner_zone_end_n,
        'throwin_end_n': throwin_end_n,
        'pre_box_end_n': pre_box_end_n,
        'third_transition_n': third_transition_n,
        'ended_final_third_n': ended_final_third_n,
        'center_circle_touch_n': center_circle_touch_n,
        'mean_forward_delta': float(np.mean(fwd_vals)) if fwd_vals else 0.0,
        'max_forward_delta': float(np.max(fwd_vals)) if fwd_vals else 0.0,
        'mean_abs_lateral_delta': float(np.mean(lat_vals)) if lat_vals else 0.0,
        'mean_gap_sec': float(np.mean(gaps)) if gaps else 0.0,
        'max_gap_sec': float(np.max(gaps)) if gaps else 0.0,
        'low_signal_n': low_signal_n,
        'class_shot_n': cc.get('shot', 0),
        'class_transition_n': cc.get('transition', 0),
        'class_turnover_n': cc.get('turnover', 0),
        'class_build_up_n': cc.get('build_up', 0),
        'class_foul_n': cc.get('foul', 0),
        'class_low_signal_n': cc.get('low_signal', 0),
        'class_meta_n': cc.get('meta', 0),
        'class_discipline_n': cc.get('discipline', 0),
        'class_duel_n': cc.get('duel', 0),
        'class_defensive_action_n': cc.get('defensive_action', 0),
        'class_set_piece_or_ref_n': cc.get('set_piece_or_ref', 0),
        'first_type': types[0] if types else None,
        'last_type': types[-1] if types else None,
        'period': int(((events[0].get('event_json') or {}).get('period') or 1)),
        't_start': t_start,
        't_end': t_end,
    }
    return row


rows = []
for i, p in enumerate(payloads_all, start=1):
    r = chain_features_from_payload(p, i)
    if r is not None:
        rows.append(r)

df_chains = pd.DataFrame(rows)

# validate event types mapping vs spec
types_data = set()
for p in payloads_all:
    for e in p.get('events', []):
        t = ((e.get('event_json') or {}).get('type') or {}).get('name')
        if t:
            types_data.add(t)

missing_in_map = sorted([t for t in types_data if t not in EVENT_CLASS_MAP])
missing_from_spec = sorted([t for t in SPEC_EVENT_TYPES_33 if t not in EVENT_CLASS_MAP])
extra_in_map = sorted([t for t in EVENT_CLASS_MAP.keys() if t not in SPEC_EVENT_TYPES_33])

print('Spec types total:', len(SPEC_EVENT_TYPES_33))
print('Map types total :', len(EVENT_CLASS_MAP))
print('Data types total:', len(types_data))
print('Missing in map (from data):', missing_in_map)
print('Missing in map (from spec):', missing_from_spec)
print('Extra in map (not in spec):', extra_in_map)

display(df_chains.head(10))
print('chains with features:', len(df_chains))

Spec types total: 33
Map types total : 34
Data types total: 33
Missing in map (from data): []
Missing in map (from spec): []
Extra in map (not in spec): ['Referee Ball-Drop']


,chain_id,match_id,stage,to_speak,events_n,duration_sec,high_event_n,set_play_n,long_pass_n,entered_opp_half_n,...,class_meta_n,class_discipline_n,class_duel_n,class_defensive_action_n,class_set_piece_or_ref_n,first_type,last_type,period,t_start,t_end
0,1,3930158,EURO 2024,1,1,0.000,0,0,0,0,...,1,0,0,0,0,Starting XI,Starting XI,1,00:00:00.000,00:00:00.000
1,2,3930158,EURO 2024,1,1,0.000,0,0,0,0,...,1,0,0,0,0,Starting XI,Starting XI,1,00:00:00.000,00:00:00.000
2,3,3930158,EURO 2024,1,2,0.000,0,0,0,0,...,2,0,0,0,0,Half Start,Half Start,1,00:00:00.000,00:00:00.000
3,4,3930158,EURO 2024,1,4,2.974,0,4,1,1,...,0,0,0,0,0,Pass,Pass,1,00:00:01.191,00:00:04.165
4,5,3930158,EURO 2024,1,1,0.000,0,1,0,0,...,0,0,0,0,0,Ball Receipt*,Ball Receipt*,1,00:00:07.397,00:00:07.397
5,6,3930158,EURO 2024,1,3,0.474,1,3,0,0,...,0,0,1,0,0,Duel,Ball Recovery,1,00:00:07.397,00:00:07.871
6,7,3930158,EURO 2024,1,6,0.851,2,6,0,0,...,0,0,0,1,0,Ball Recovery,Block,1,00:00:08.713,00:00:09.564
7,8,3930158,EURO 2024,1,1,0.000,1,1,0,0,...,0,0,0,1,0,Block,Block,1,00:00:10.048,00:00:10.048
8,9,3930158,EURO 2024,1,1,0.000,1,1,0,0,...,0,0,0,1,0,Clearance,Clearance,1,00:00:10.987,00:00:10.987
9,10,3930158,EURO 2024,1,6,3.184,1,6,0,1,...,0,0,0,0,0,Ball Recovery,Pass,1,00:00:13.928,00:00:17.112


chains with features: 49300


## Методология: soft-labels без ручной разметки

У нас пока нет полноценной ручной gold-разметки: никто не отметил для каждой цепочки, надо ли её комментировать. Поэтому используем **soft-labels** — эвристическую proxy-разметку.

Это не “истина”, а способ обучить и сравнить модели отбора эпизодов.

Шкала 5 классов:

- `0` — почти точно пропустить: служебное/низкосигнальное/малозначимое;
- `1` — слабый эпизод: можно пропустить, если рядом есть более важные;
- `2` — умеренно важный build-up: краткий комментарий возможен;
- `3` — важный эпизод: переход владения, стандарт, продвижение, вход в опасную зону;
- `4` — обязательно комментировать: удар, опасный момент, вход в штрафную, выраженный исход.

Для практического сценария LLM дополнительно делаем 3 класса:

- `0 = skip` — не отправлять в LLM;
- `1 = brief` — можно отправить, нужен короткий комментарий;
- `2 = must` — отправлять обязательно.

Почему не сразу 2 класса: футбольные эпизоды не бинарные. Есть много “середины”: владение развивается, но ещё не опасно. 3-классная схема лучше соответствует будущему пайплайну: пропустить / коротко описать / обязательно описать.


In [26]:
def soft_label_rule_5(r):
    score = 0
    score += 3 * int(r['class_shot_n'] > 0)
    score += 2 * int(r['high_event_n'] > 0)
    score += 2 * int(r['entered_opp_box_n'] > 0)
    score += 1 * int(r['long_pass_n'] > 0)
    score += 1 * int(r['forward_moves_n'] > 0)
    score += 1 * int(r['class_transition_n'] > 0)
    score += 1 * int(r['switched_flank_n'] > 0)
    score += 1 * int(r['bad_pass_outcome_n'] > 0)
    score -= 1 * int(r['low_signal_n'] >= max(2, r['events_n'] // 2))
    if r['events_n'] >= 10 and r['high_event_n'] == 0 and r['entered_opp_box_n'] == 0:
        score -= 1

    if score <= 0:
        return 0
    if score <= 2:
        return 1
    if score <= 4:
        return 2
    if score <= 6:
        return 3
    return 4


def map_5_to_3(lbl5):
    if lbl5 <= 1:
        return 0
    if lbl5 <= 3:
        return 1
    return 2


df_chains['soft_label_5'] = df_chains.apply(soft_label_rule_5, axis=1)
df_chains['soft_label_3'] = df_chains['soft_label_5'].apply(map_5_to_3)
df_chains['soft_label'] = df_chains['soft_label_5']  # backward compatibility

print('soft_label_5 dist:', df_chains['soft_label_5'].value_counts().sort_index().to_dict())
print('soft_label_3 dist:', df_chains['soft_label_3'].value_counts().sort_index().to_dict())

soft_label_5 dist: {0: 3911, 1: 25429, 2: 13930, 3: 4613, 4: 1417}
soft_label_3 dist: {0: 29340, 1: 18543, 2: 1417}


## Что делаем в этой ячейке
Показываем, как перевести весь JSON цепочек в DataFrame “на уровне события внутри цепочки”.

Это отвечает на твой вопрос: да, можно разворачивать цепочки в табличный формат.

In [27]:
event_level_rows = []
for p in payloads_all:
    cid = p.get('chain_event_ids')
    for order, it in enumerate(p.get('events', []), start=1):
        ej = it.get('event_json') or {}
        dr = it.get('derived') or {}
        event_level_rows.append({
            'match_id': p.get('match_id'),
            'stage': p.get('stage'),
            'chain_id': str(cid),
            'order_in_chain': order,
            'event_id': it.get('event_id'),
            'period': ej.get('period'),
            'timestamp': ej.get('timestamp'),
            'type_name': ((ej.get('type') or {}).get('name')),
            'team_name': ((ej.get('team') or {}).get('name')),
            'player_name': ((ej.get('player') or {}).get('name')),
            'start_abs': (dr.get('zones') or {}).get('start_abs'),
            'end_abs': (dr.get('zones') or {}).get('end_abs'),
            'start_rel': (dr.get('zones') or {}).get('start_rel'),
            'end_rel': (dr.get('zones') or {}).get('end_rel'),
            'movement_label': (dr.get('movement') or {}).get('label'),
            'forward_delta': (dr.get('movement') or {}).get('forward_delta'),
            'is_long_pass': (dr.get('episode_signals') or {}).get('is_long_pass'),
            'entered_opponent_box': (dr.get('episode_signals') or {}).get('entered_opponent_box'),
            'sb360_is_null': int(it.get('sb360_json') is None),
        })

df_chain_events = pd.DataFrame(event_level_rows)
print('event-level rows:', len(df_chain_events))
display(df_chain_events.head(12))

event-level rows: 187858


,match_id,stage,chain_id,order_in_chain,event_id,period,timestamp,type_name,team_name,player_name,start_abs,end_abs,start_rel,end_rel,movement_label,forward_delta,is_long_pass,entered_opponent_box,sb360_is_null
0,3930158,EURO 2024,['f2207d37-beed-4073-84a5-f9d696e2afdd'],1,f2207d37-beed-4073-84a5-f9d696e2afdd,1,00:00:00.000,Starting XI,Germany,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,3930158,EURO 2024,['43e494fe-4871-4b2a-a509-a325c17a5e3f'],1,43e494fe-4871-4b2a-a509-a325c17a5e3f,1,00:00:00.000,Starting XI,Scotland,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,3930158,EURO 2024,"['d9de8fe7-3122-4067-a1c2-a742c7edb2b8', '455f819f-3926-4ed4-a0e9-4e9e4d0bf95a']",1,d9de8fe7-3122-4067-a1c2-a742c7edb2b8,1,00:00:00.000,Half Start,Germany,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,3930158,EURO 2024,"['d9de8fe7-3122-4067-a1c2-a742c7edb2b8', '455f819f-3926-4ed4-a0e9-4e9e4d0bf95a']",2,455f819f-3926-4ed4-a0e9-4e9e4d0bf95a,1,00:00:00.000,Half Start,Scotland,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,3930158,EURO 2024,"['df6f6fb7-f0b0-4561-9171-f4172f5e97e7', 'f9195e75-1617-478c-bf9d-66f17151724a', '65072c25-9c61-441e-a51c-52856c51ca...",1,df6f6fb7-f0b0-4561-9171-f4172f5e97e7,1,00:00:01.191,Pass,Germany,Havertz,center_circle_zone,right_half:center_lane,center_line,own_half,backward,-14.3,0.0,0.0,0
5,3930158,EURO 2024,"['df6f6fb7-f0b0-4561-9171-f4172f5e97e7', 'f9195e75-1617-478c-bf9d-66f17151724a', '65072c25-9c61-441e-a51c-52856c51ca...",2,f9195e75-1617-478c-bf9d-66f17151724a,1,00:00:02.426,Ball Receipt*,Germany,Mittelstädt,right_half:center_lane,NaN,own_half,NaN,unknown,NaN,0.0,0.0,0
6,3930158,EURO 2024,"['df6f6fb7-f0b0-4561-9171-f4172f5e97e7', 'f9195e75-1617-478c-bf9d-66f17151724a', '65072c25-9c61-441e-a51c-52856c51ca...",3,65072c25-9c61-441e-a51c-52856c51ca94,1,00:00:02.426,Carry,Germany,Mittelstädt,right_half:center_lane,right_half:center_lane,own_half,own_half,forward,3.6,0.0,0.0,0
7,3930158,EURO 2024,"['df6f6fb7-f0b0-4561-9171-f4172f5e97e7', 'f9195e75-1617-478c-bf9d-66f17151724a', '65072c25-9c61-441e-a51c-52856c51ca...",4,01be895c-5835-4a8d-b99f-2a642f5d26dc,1,00:00:04.165,Pass,Germany,Mittelstädt,right_half:center_lane,pre_box_left,own_half,opponent_half,forward,50.6,1.0,0.0,0
8,3930158,EURO 2024,['9aa6c033-3015-4e13-9566-bd1913f754d3'],1,9aa6c033-3015-4e13-9566-bd1913f754d3,1,00:00:07.397,Ball Receipt*,Germany,Havertz,pre_box_left,NaN,opponent_half,NaN,unknown,NaN,0.0,0.0,0
9,3930158,EURO 2024,"['d565ae37-8197-4510-b01f-171a04435ec1', '859a97e2-e05a-42c3-a020-32a8d7848300', '1b105dba-9074-4a8d-b6cd-ab686d59d3...",1,d565ae37-8197-4510-b01f-171a04435ec1,1,00:00:07.397,Duel,Scotland,Hendry,pre_box_left,NaN,own_half,NaN,unknown,NaN,0.0,0.0,1


## Методология: обучение моделей отбора эпизодов

Здесь мы обучаем модели предсказывать `soft_label_3` по признакам цепочки.

Это **не оценка качества LLM-комментария**. Это отдельная задача: научиться автоматически выбирать, какие эпизоды стоит отправлять в LLM.

Модели:

- **Logistic Regression** — простой линейный baseline для многоклассовой классификации;
- **CatBoostClassifier** — основной табличный метод, хорошо работает с числовыми и категориальными признаками;
- **TabNet** — нейросетевой табличный baseline, если установлен.

Разбиение делаем **по матчам**, а не случайно по цепочкам. Это важно: иначе события одного матча попадут и в train, и в test, и оценка будет слишком оптимистичной.

Метрики:

- `macro_f1` — главная: одинаково учитывает каждый класс, даже если класс `must` редкий;
- `weighted_f1` — показывает качество с учётом частоты классов;
- `balanced_accuracy` — средняя полнота по классам;
- `accuracy` — вспомогательная, но не главная, потому что классы несбалансированы.

Ответ на вопрос “нужны ли зоны CatBoost”: да. CatBoost не видит поле как картинку, но видит признаки вроде `entered_opp_box_n`, `third_transition_n`, `ended_final_third_n`, `mean_forward_delta`, `start/end zones`. Это как раз футбольная геометрия в табличном виде.


In [28]:
TARGET = 'soft_label_3'
feature_cols_num = [
    'events_n', 'duration_sec', 'high_event_n', 'set_play_n', 'long_pass_n',
    'entered_opp_half_n', 'entered_opp_box_n', 'left_own_box_n',
    'forward_moves_n', 'backward_moves_n', 'lateral_moves_n',
    'progressive_carry_n', 'switched_flank_n', 'bad_pass_outcome_n',
    'corner_zone_end_n', 'throwin_end_n', 'pre_box_end_n',
    'third_transition_n', 'ended_final_third_n', 'center_circle_touch_n',
    'mean_forward_delta', 'max_forward_delta', 'mean_abs_lateral_delta',
    'mean_gap_sec', 'max_gap_sec', 'low_signal_n',
    'class_shot_n', 'class_transition_n', 'class_turnover_n', 'class_build_up_n',
    'class_foul_n', 'class_low_signal_n', 'class_meta_n', 'class_discipline_n',
    'class_duel_n', 'class_defensive_action_n', 'class_set_piece_or_ref_n',
]
feature_cols_cat = ['first_type', 'last_type', 'period', 'stage']

work = df_chains.dropna(subset=[TARGET]).copy()
work['match_id'] = work['match_id'].astype(str)
for c in feature_cols_cat:
    work[c] = work[c].fillna('unknown').astype(str)

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
trainval_idx, test_idx = next(gss.split(work, groups=work['match_id']))
trainval_df = work.iloc[trainval_idx].copy()
test_df = work.iloc[test_idx].copy()

val_rel = VAL_SIZE / (1.0 - TEST_SIZE)
gss2 = GroupShuffleSplit(n_splits=1, test_size=val_rel, random_state=RANDOM_STATE + 1)
train_idx, val_idx = next(gss2.split(trainval_df, groups=trainval_df['match_id']))
train_df = trainval_df.iloc[train_idx].copy()
val_df = trainval_df.iloc[val_idx].copy()

print('train/val/test rows:', len(train_df), len(val_df), len(test_df))
print('match split:', train_df['match_id'].nunique(), val_df['match_id'].nunique(), test_df['match_id'].nunique())


def evaluate(y_true, y_pred):
    return {
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted'),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
    }

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), feature_cols_num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), feature_cols_cat),
])

logreg = Pipeline([
    ('prep', prep),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
])
logreg.fit(train_df[feature_cols_num + feature_cols_cat], train_df[TARGET])

leader_rows = []
for split_name, sdf in [('val', val_df), ('test', test_df)]:
    pred = logreg.predict(sdf[feature_cols_num + feature_cols_cat])
    m = evaluate(sdf[TARGET], pred)
    leader_rows.append({'model':'LogReg','split':split_name, **m})

if HAS_CATBOOST:
    cat_features_idx = [len(feature_cols_num) + i for i in range(len(feature_cols_cat))]
    cb = CatBoostClassifier(
        loss_function='MultiClass', eval_metric='TotalF1', random_seed=RANDOM_STATE,
        depth=8, learning_rate=0.06, iterations=600, verbose=False
    )
    Xtr = train_df[feature_cols_num + feature_cols_cat].copy()
    Xva = val_df[feature_cols_num + feature_cols_cat].copy()
    Xte = test_df[feature_cols_num + feature_cols_cat].copy()
    for c in feature_cols_cat:
        Xtr[c] = Xtr[c].fillna('unknown').astype(str)
        Xva[c] = Xva[c].fillna('unknown').astype(str)
        Xte[c] = Xte[c].fillna('unknown').astype(str)

    cb.fit(Xtr, train_df[TARGET], cat_features=cat_features_idx, eval_set=(Xva, val_df[TARGET]), use_best_model=True)

    for split_name, Xs, ys in [('val', Xva, val_df[TARGET]), ('test', Xte, test_df[TARGET])]:
        pred = cb.predict(Xs)
        pred = np.array(pred).reshape(-1).astype(int)
        m = evaluate(ys, pred)
        leader_rows.append({'model':'CatBoost','split':split_name, **m})

if HAS_TABNET:
    Xtr = train_df[feature_cols_num + feature_cols_cat].copy()
    Xva = val_df[feature_cols_num + feature_cols_cat].copy()
    Xte = test_df[feature_cols_num + feature_cols_cat].copy()

    for c in feature_cols_cat:
        Xtr[c] = Xtr[c].fillna('unknown').astype(str)
        Xva[c] = Xva[c].fillna('unknown').astype(str)
        Xte[c] = Xte[c].fillna('unknown').astype(str)
        allv = pd.Categorical(pd.concat([Xtr[c], Xva[c], Xte[c]], axis=0))
        mp = {k: i for i, k in enumerate(allv.categories)}
        Xtr[c] = Xtr[c].map(mp).fillna(0).astype(int)
        Xva[c] = Xva[c].map(mp).fillna(0).astype(int)
        Xte[c] = Xte[c].map(mp).fillna(0).astype(int)

    tabnet = TabNetClassifier(seed=RANDOM_STATE, verbose=0)
    tabnet.fit(
        X_train=Xtr.values, y_train=train_df[TARGET].astype(int).values,
        eval_set=[(Xva.values, val_df[TARGET].astype(int).values)],
        max_epochs=120, patience=20, batch_size=1024, virtual_batch_size=128,
    )

    for split_name, Xs, ys in [('val', Xva.values, val_df[TARGET].astype(int).values), ('test', Xte.values, test_df[TARGET].astype(int).values)]:
        pred = tabnet.predict(Xs).astype(int)
        m = evaluate(ys, pred)
        leader_rows.append({'model':'TabNet','split':split_name, **m})

leaderboard_df = pd.DataFrame(leader_rows).sort_values(['split','macro_f1'], ascending=[True, False]).reset_index(drop=True)
display(leaderboard_df)

train/val/test rows: 28740 9779 10781
match split: 30 10 11

Early stopping occurred at epoch 80 with best_epoch = 60 and best_val_0_accuracy = 0.99949


,model,split,macro_f1,weighted_f1,balanced_accuracy,accuracy
0,CatBoost,test,0.997754,0.999630,0.999676,0.999629
1,TabNet,test,0.996363,0.998422,0.994781,0.998423
2,LogReg,test,0.911630,0.959168,0.960870,0.958260
3,TabNet,val,0.997997,0.999489,0.998557,0.999489
4,CatBoost,val,0.997540,0.999592,0.999631,0.999591
5,LogReg,val,0.905294,0.957217,0.960777,0.956028


## Дополнительный эксперимент: ранжирование эпизодов

Классификация отвечает на вопрос: **какой класс важности у цепочки** (`skip / brief / must`).

Ранжирование отвечает на другой вопрос: **какие эпизоды внутри матча самые важные и должны попасть в ограниченный бюджет комментариев**.

Это полезно, потому что в реальном сервисе у нас есть ограничения:

- нельзя комментировать каждое событие;
- LLM-токены дорогие;
- озвучка не должна перекрывать весь матч;
- если рядом есть несколько эпизодов, лучше выбрать наиболее важный.

Поэтому ранжирование не заменяет классификацию, а дополняет её.

Как оцениваем ранжирование без gold-разметки:

- используем `soft_label_5` как proxy-релевантность;
- группируем объекты по `match_id`;
- модель должна ставить более важные цепочки выше внутри каждого матча;
- считаем `NDCG@K`, `Precision@K`, `Recall@K`.

Интерпретация:

- `Precision@K`: какая доля top-K эпизодов действительно относится к важным (`soft_label_3 >= 1` или `soft_label_3 == 2`);
- `Recall@K`: какую долю важных эпизодов мы поймали в top-K;
- `NDCG@K`: насколько хорошо модель упорядочила эпизоды с учётом градации важности.

Для ВКР это можно подать как отдельный сценарий: **не только классифицировать эпизоды, но и ранжировать их для ограниченного бюджета LLM/озвучки**.


In [ ]:
# --- Ranking experiment ---
# Этот блок опциональный: он показывает, как выбрать top-K эпизодов внутри каждого матча.

RANK_TARGET = 'soft_label_5'
RANK_K_LIST = [10, 25, 50, 100]


def _dcg_at_k(relevance, k):
    rel = np.asarray(relevance, dtype=float)[:k]
    if len(rel) == 0:
        return 0.0
    gains = (2 ** rel - 1)
    discounts = np.log2(np.arange(len(rel)) + 2)
    return float(np.sum(gains / discounts))


def ndcg_at_k(y_true, y_score, k):
    order = np.argsort(-np.asarray(y_score))
    ideal = np.argsort(-np.asarray(y_true))
    dcg = _dcg_at_k(np.asarray(y_true)[order], k)
    idcg = _dcg_at_k(np.asarray(y_true)[ideal], k)
    return 0.0 if idcg == 0 else dcg / idcg


def precision_recall_at_k(y_true_binary, y_score, k):
    y = np.asarray(y_true_binary).astype(int)
    if len(y) == 0:
        return 0.0, 0.0
    order = np.argsort(-np.asarray(y_score))[:min(k, len(y))]
    hits = int(y[order].sum())
    precision = hits / max(len(order), 1)
    recall = hits / max(int(y.sum()), 1)
    return precision, recall


def evaluate_ranking_by_match(df, score_col, relevance_col='soft_label_5', important_col='soft_label_3'):
    rows = []
    for k in RANK_K_LIST:
        ndcgs, ps, rs = [], [], []
        for mid, g in df.groupby('match_id'):
            if len(g) == 0:
                continue
            y_rel = g[relevance_col].astype(float).values
            y_score = g[score_col].astype(float).values
            y_imp = (g[important_col].astype(int).values >= 1).astype(int)
            ndcgs.append(ndcg_at_k(y_rel, y_score, k))
            p_at, r_at = precision_recall_at_k(y_imp, y_score, k)
            ps.append(p_at)
            rs.append(r_at)
        rows.append({
            'score': score_col,
            'k': k,
            'ndcg_at_k': float(np.mean(ndcgs)) if ndcgs else np.nan,
            'precision_at_k': float(np.mean(ps)) if ps else np.nan,
            'recall_at_k': float(np.mean(rs)) if rs else np.nan,
        })
    return rows

ranking_rows = []

# Baseline 1: эвристический score, по которому строились soft-labels.
# Он нужен как sanity-check и верхняя proxy-граница, но это не честная независимая модель.
df_rank_test = test_df.copy()
df_rank_test['score_rule'] = df_rank_test['importance_score_rule'].astype(float)
ranking_rows += [{'model': 'RuleScore', **r} for r in evaluate_ranking_by_match(df_rank_test, 'score_rule')]

# Baseline 2: вероятность класса must/brief из LogReg.
try:
    logreg_proba = logreg.predict_proba(test_df[feature_cols_num + feature_cols_cat])
    logreg_classes = list(logreg.named_steps['clf'].classes_)
    score = np.zeros(len(test_df), dtype=float)
    for cls_weight, cls in [(1.0, 1), (2.0, 2)]:
        if cls in logreg_classes:
            score += cls_weight * logreg_proba[:, logreg_classes.index(cls)]
    df_rank_lr = test_df.copy()
    df_rank_lr['score_logreg'] = score
    ranking_rows += [{'model': 'LogRegRankByProba', **r} for r in evaluate_ranking_by_match(df_rank_lr, 'score_logreg')]
except Exception as e:
    print('LogReg ranking skipped:', e)

# CatBoostRanker: учим ранжировать цепочки внутри матча.
if HAS_CATBOOST:
    try:
        rank_features = feature_cols_num + feature_cols_cat
        Xtr = train_df[rank_features].copy()
        Xte = test_df[rank_features].copy()
        for c in feature_cols_cat:
            Xtr[c] = Xtr[c].fillna('unknown').astype(str)
            Xte[c] = Xte[c].fillna('unknown').astype(str)

        train_sorted = train_df.reset_index(drop=True).copy()
        test_sorted = test_df.reset_index(drop=True).copy()
        Xtr = Xtr.reset_index(drop=True)
        Xte = Xte.reset_index(drop=True)

        # CatBoostRanker требует, чтобы строки одной группы шли подряд.
        order_tr = train_sorted.sort_values(['match_id', 'chain_id']).index
        order_te = test_sorted.sort_values(['match_id', 'chain_id']).index
        Xtr_s = Xtr.loc[order_tr].reset_index(drop=True)
        ytr_s = train_sorted.loc[order_tr, RANK_TARGET].astype(float).reset_index(drop=True)
        group_tr = train_sorted.loc[order_tr, 'match_id'].astype(str).reset_index(drop=True)

        Xte_s = Xte.loc[order_te].reset_index(drop=True)
        test_s = test_sorted.loc[order_te].reset_index(drop=True)

        cat_features_idx = [len(feature_cols_num) + i for i in range(len(feature_cols_cat))]
        ranker = CatBoostRanker(
            loss_function='YetiRank',
            iterations=400,
            depth=6,
            learning_rate=0.05,
            random_seed=RANDOM_STATE,
            verbose=False,
        )
        ranker.fit(Xtr_s, ytr_s, group_id=group_tr, cat_features=cat_features_idx)
        test_s['score_catboost_ranker'] = ranker.predict(Xte_s)
        ranking_rows += [{'model': 'CatBoostRanker', **r} for r in evaluate_ranking_by_match(test_s, 'score_catboost_ranker')]
    except Exception as e:
        print('CatBoostRanker skipped:', e)

ranking_leaderboard_df = pd.DataFrame(ranking_rows)
if not ranking_leaderboard_df.empty:
    display(ranking_leaderboard_df.sort_values(['k', 'ndcg_at_k'], ascending=[True, False]))
else:
    print('No ranking results')


## Таблица признаков для ВКР

В этой ячейке создаём словарь признаков: название, тип, описание, доля пропусков и факт использования в обучении.

Эта таблица нужна для методологической части ВКР: по ней видно, что модель обучается не на “магии”, а на интерпретируемых футбольных признаках.

Группы признаков:

- размер и длительность цепочки;
- типы и классы событий;
- пространственные признаки;
- направление движения;
- признаки паса;
- признаки исхода;
- временные гэпы;
- категориальные признаки первого/последнего события, тайма и стадии матча.


In [ ]:
FEATURE_DESCRIPTIONS = {
    'events_n': 'Количество событий в цепочке',
    'duration_sec': 'Длительность цепочки, сек',
    'high_event_n': 'Количество high-impact событий',
    'set_play_n': 'Количество событий со стандартов',
    'long_pass_n': 'Количество длинных передач',
    'entered_opp_half_n': 'Входы на чужую половину',
    'entered_opp_box_n': 'Входы в чужую штрафную',
    'left_own_box_n': 'Выходы из своей штрафной',
    'forward_moves_n': 'Движения вперед (по forward_delta)',
    'backward_moves_n': 'Движения назад',
    'lateral_moves_n': 'Поперечные смещения',
    'progressive_carry_n': 'Прогрессивные ведения',
    'switched_flank_n': 'Переводы фланга',
    'bad_pass_outcome_n': 'Пасы с негативным outcome',
    'corner_zone_end_n': 'Окончания в угловой зоне',
    'throwin_end_n': 'Окончания у боковой (throw-in zone)',
    'pre_box_end_n': 'Окончания перед штрафной',
    'third_transition_n': 'Переходы между третями поля',
    'ended_final_third_n': 'Окончания в финальной трети атаки',
    'center_circle_touch_n': 'Касания в центральном круге',
    'mean_forward_delta': 'Средний прогресс к воротам',
    'max_forward_delta': 'Максимальный прогресс к воротам',
    'mean_abs_lateral_delta': 'Среднее поперечное смещение |dy|',
    'mean_gap_sec': 'Средний временной гэп между событиями',
    'max_gap_sec': 'Макс. временной гэп',
    'low_signal_n': 'Low-signal события в цепочке',
    'class_shot_n': 'События класса shot',
    'class_transition_n': 'События класса transition',
    'class_turnover_n': 'События класса turnover',
    'class_build_up_n': 'События класса build_up',
    'class_foul_n': 'События класса foul',
    'class_low_signal_n': 'События класса low_signal',
    'class_meta_n': 'События класса meta',
    'class_discipline_n': 'События класса discipline',
    'class_duel_n': 'События класса duel',
    'class_defensive_action_n': 'События класса defensive_action',
    'class_set_piece_or_ref_n': 'События класса set_piece_or_ref',
    'first_type': 'Тип первого события цепочки',
    'last_type': 'Тип последнего события цепочки',
    'period': 'Тайм цепочки',
    'stage': 'Стадия турнира',
    'soft_label_5': 'Soft-label 0..4 (эвристика)',
    'soft_label_3': 'Soft-label 0..2 (агрегация 5->3)',
}

rows = []
all_train_features = set(feature_cols_num + feature_cols_cat)
for col in sorted(set(feature_cols_num + feature_cols_cat + [TARGET, 'soft_label_5'])):
    dtype = 'categorical' if col in feature_cols_cat else 'numeric'
    if col in {TARGET, 'soft_label_5'}:
        dtype = 'target'

    rows.append({
        'feature': col,
        'dtype': dtype,
        'used_in_training': int(col in all_train_features),
        'missing_rate': float(work[col].isna().mean()) if col in work.columns else np.nan,
        'description': FEATURE_DESCRIPTIONS.get(col, ''),
    })

feature_table_df = pd.DataFrame(rows).sort_values(['used_in_training', 'dtype', 'feature'], ascending=[False, True, True]).reset_index(drop=True)

print('features used in model:', len(all_train_features))
print('numeric               :', len(feature_cols_num))
print('categorical           :', len(feature_cols_cat))
display(feature_table_df)


## Экспорт артефактов

Сохраняем всё, что понадобится дальше:

- `chains_features_softlabels.csv` — таблица цепочек, признаков и soft-labels;
- `leaderboard_val_test.csv` — сравнение моделей;
- `feature_dictionary_detailed.csv` — подробный словарь признаков для ВКР;
- `llm_payloads_softlabel3_filtered_all51.json` — payload для LLM по всем 51 матчам;
- `llm_payloads_softlabel3_filtered_top15.json` — payload только для выбранных 15 матчей;
- отдельные `json/jsonl` по каждому матчу.

Важно: LLM получает не все цепочки, а только те, где `soft_label_3 >= MIN_SOFT_LABEL_FOR_LLM`. Это экономит токены и уменьшает количество бессодержательных комментариев.


In [ ]:
TAB_DIR = OUT_DIR / 'tables'
TAB_DIR.mkdir(parents=True, exist_ok=True)
PAYLOAD_DIR = OUT_DIR / 'payloads_by_match'
PAYLOAD_DIR.mkdir(parents=True, exist_ok=True)

# features table
feat_path = OUT_DIR / 'chains_features_softlabels.csv'
df_chains.to_csv(feat_path, index=False)

# leaderboard
leader_path = TAB_DIR / 'leaderboard_val_test.csv'
leaderboard_df.to_csv(leader_path, index=False)

# ranking leaderboard (если запускали ranking-блок)
if 'ranking_leaderboard_df' in globals() and isinstance(ranking_leaderboard_df, pd.DataFrame):
    ranking_path = TAB_DIR / 'ranking_leaderboard_test.csv'
    ranking_leaderboard_df.to_csv(ranking_path, index=False)

# feature dictionary (подробная)
feature_table_path = TAB_DIR / 'feature_dictionary_detailed.csv'
feature_table_df.to_csv(feature_table_path, index=False)

# legacy short dictionary
feature_dict = pd.DataFrame([
    {'feature': c, 'kind': 'numeric'} for c in feature_cols_num
] + [
    {'feature': c, 'kind': 'categorical'} for c in feature_cols_cat
])
feature_dict.to_csv(TAB_DIR / 'feature_dictionary.csv', index=False)

# chain selection for LLM
llm_keep = set(df_chains.loc[df_chains['soft_label_3'] >= MIN_SOFT_LABEL_FOR_LLM, 'chain_id'].tolist())
match_to_speak = {str(r['match_id']): int(r.get('to_speak', 0)) for _, r in manifest.iterrows()}

payload_rows = []
for idx, p in enumerate(payloads_all, start=1):
    if idx not in llm_keep:
        continue
    payload_rows.append(p)

all51_path = OUT_DIR / 'llm_payloads_softlabel3_filtered_all51.json'
all51_path.write_text(json.dumps(payload_rows, ensure_ascii=False, indent=2), encoding='utf-8')

payload_rows_top15 = [p for p in payload_rows if match_to_speak.get(str(p.get('match_id')), 0) == 1]
top15_path = OUT_DIR / 'llm_payloads_softlabel3_filtered_top15.json'
top15_path.write_text(json.dumps(payload_rows_top15, ensure_ascii=False, indent=2), encoding='utf-8')

# split payloads by match
by_match_all = {}
by_match_top = {}
for p in payload_rows:
    mid = str(p.get('match_id'))
    by_match_all.setdefault(mid, []).append(p)
    if match_to_speak.get(mid, 0) == 1:
        by_match_top.setdefault(mid, []).append(p)

rows_all = []
for mid, arr in sorted(by_match_all.items()):
    jpath = PAYLOAD_DIR / f'{mid}_llm_payloads_softlabel3_filtered_all51.json'
    jlpath = PAYLOAD_DIR / f'{mid}_llm_payloads_softlabel3_filtered_all51.jsonl'
    jpath.write_text(json.dumps(arr, ensure_ascii=False, indent=2), encoding='utf-8')
    with jlpath.open('w', encoding='utf-8') as f:
        for row in arr:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    rows_all.append({'match_id': mid, 'chains_saved': len(arr), 'json': str(jpath), 'jsonl': str(jlpath)})

rows_top = []
for mid, arr in sorted(by_match_top.items()):
    jpath = PAYLOAD_DIR / f'{mid}_llm_payloads_softlabel3_filtered_top15.json'
    jlpath = PAYLOAD_DIR / f'{mid}_llm_payloads_softlabel3_filtered_top15.jsonl'
    jpath.write_text(json.dumps(arr, ensure_ascii=False, indent=2), encoding='utf-8')
    with jlpath.open('w', encoding='utf-8') as f:
        for row in arr:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    rows_top.append({'match_id': mid, 'chains_saved': len(arr), 'json': str(jpath), 'jsonl': str(jlpath)})

pd.DataFrame(rows_all).to_csv(PAYLOAD_DIR / 'index_all51.csv', index=False)
pd.DataFrame(rows_top).to_csv(PAYLOAD_DIR / 'index_top15.csv', index=False)

print('saved:', feat_path)
print('saved:', all51_path)
print('saved:', top15_path)
print('saved:', PAYLOAD_DIR / 'index_all51.csv')
print('saved:', PAYLOAD_DIR / 'index_top15.csv')
print('saved:', feature_table_path)
if 'ranking_path' in globals():
    print('saved:', ranking_path)

## Итоговая методология для ВКР

Фиксируем краткое описание эксперимента в JSON: откуда берём данные, как формируем цепочки, какие признаки используем, как задаём soft-labels, какие модели и метрики сравниваем.

Этот файл можно использовать как “паспорт эксперимента”: если позже поменяется порог цепочек или схема soft-labels, это будет явно видно.


In [ ]:
methodology = {
    'data_source': str(PROCESSED_DIR),
    'input_artifacts': {
        'events_std': 'стандартизированные события после half-aware flip',
        'events_std_clean': 'очищенные события для LLM payload',
        'sb360_std': 'стандартизированный StatsBomb360',
        'bad_ids': 'события, для которых 360 намеренно отключается',
        'llm_items_jsonl': 'предыдущая версия LLM items; в этом ноутбуке payload пересобирается компактными цепочками',
    },
    'zone_logic': {
        'implemented_in': 'statsbomb_toolkit.sber_exports.preprocessing.build_chain_payload_v4',
        'absolute_zones': 'zone_label_abs: геометрия поля 120x80',
        'team_relative_zones': 'zone_label_team: своя/чужая зона с учетом ref_by_period',
        'movement': 'forward_delta > 0 означает движение к чужим воротам команды с мячом',
    },
    'chain_methodology': {
        'base': 'undirected related_events connected components',
        'max_events_per_chain': MAX_EVENTS_PER_CHAIN,
        'max_chain_duration_sec': MAX_CHAIN_DURATION_SEC,
        'max_gap_sec': MAX_GAP_SEC,
        'reason': 'не отдавать LLM слишком длинные related-компоненты',
    },
    'features': {
        'numeric_n': len(feature_cols_num),
        'categorical_n': len(feature_cols_cat),
        'feature_table': str(TAB_DIR / 'feature_dictionary_detailed.csv') if 'TAB_DIR' in globals() else None,
    },
    'labels': {
        'soft_label_5': 'rule-based 0..4 importance',
        'soft_label_3': '0=skip, 1=brief, 2=must',
        'note': 'soft labels are proxy labels, not human gold labels',
    },
    'models': ['LogReg', 'CatBoostClassifier', 'TabNet'],
    'ranking': {
        'enabled_as_optional_experiment': True,
        'target': 'soft_label_5 relevance within match',
        'metrics': ['NDCG@K', 'Precision@K', 'Recall@K'],
        'models': ['RuleScore', 'LogReg probability ranking', 'CatBoostRanker'],
    },
    'metrics': ['macro_f1', 'weighted_f1', 'balanced_accuracy', 'accuracy'],
    'llm_export': {
        'filter': f'soft_label_3 >= {MIN_SOFT_LABEL_FOR_LLM}',
        'all51': str(OUT_DIR / 'llm_payloads_softlabel3_filtered_all51.json'),
        'top15': str(OUT_DIR / 'llm_payloads_softlabel3_filtered_top15.json'),
    },
}

methodology_path = OUT_DIR / 'methodology_summary.json'
methodology_path.write_text(json.dumps(methodology, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved:', methodology_path)
methodology
